# SBOM-to-Audit — Stage 6 matched-baseline Colab checkpoint

This checkpoint independently reproduces the controlled comparison between the temporal evidence-orchestration artefact and the frozen structured-but-unorchestrated PSIRT worksheet baseline. It uses the same source bytes and event chronology for both workflows, runs the complete release gate in an isolated environment, preserves the authoritative historical EPSS verification, and packages the comparison outputs and paper assets with the exact Git commit.

The baseline is a deterministic computational proxy. This checkpoint does not measure human analyst time, legal correctness, cognitive workload, or industrial effectiveness.


In [ ]:
REPO_URL = "https://github.com/richietrap/sbom_to_audit.git"
REF = "main"  # Replace with a Stage 6 tag after one exists.
print("Repository:", REPO_URL)
print("Reference:", REF)


In [ ]:
import os
import platform
import shutil
import subprocess
import sys
from pathlib import Path

WORKDIR = Path("/content/sbom_to_audit")
VENV = Path("/content/sbom_to_audit_stage6_venv")
for path in (WORKDIR, VENV):
    if path.exists():
        shutil.rmtree(path)
subprocess.run(
    ["git", "clone", "--depth", "1", "--branch", REF, REPO_URL, str(WORKDIR)],
    check=True,
)
os.chdir(WORKDIR)
COMMIT = subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip()
print("Tested Git commit:", COMMIT)
print("Kernel Python:", sys.version)
print("Platform:", platform.platform())


In [ ]:
try:
    subprocess.run([sys.executable, "-m", "venv", str(VENV)], check=True)
except subprocess.CalledProcessError:
    subprocess.run([sys.executable, "-m", "pip", "install", "virtualenv"], check=True)
    subprocess.run([sys.executable, "-m", "virtualenv", str(VENV)], check=True)
PYTHON = VENV / "bin" / "python"
subprocess.run(
    [str(PYTHON), "-m", "pip", "install", "--upgrade", "pip", "setuptools", "wheel"],
    check=True,
)
subprocess.run(
    [str(PYTHON), "-m", "pip", "install", "--no-cache-dir", "-e", ".[dev]"],
    check=True,
)
subprocess.run([str(PYTHON), "-m", "pip", "check"], check=True)
print("PASS: isolated dependency integrity")


In [ ]:
# Retain the mandatory authoritative historical EPSS gate from Stage 5.5.2.
import json
EPSS_DIR = Path("/content/stage6_epss_authoritative_evidence")
EPSS_REPORT = EPSS_DIR / "historical_epss_verification.json"
if EPSS_DIR.exists():
    shutil.rmtree(EPSS_DIR)
subprocess.run(
    [
        str(PYTHON), "scripts/verify_historical_epss.py", "--online",
        "--output-dir", str(EPSS_DIR), "--report", str(EPSS_REPORT),
    ],
    check=True,
)
verification = json.loads(EPSS_REPORT.read_text(encoding="utf-8"))
assert verification["status"] == "authoritative_dual_source_verified"
assert verification["api_record"]["epss"] == "0.00371"
assert verification["archive_record"]["percentile"] == "0.72343"
print("PASS: authoritative dual-source historical EPSS verification")


In [ ]:
# Execute the canonical release gate, including deterministic Stage 6 comparison.
RELEASE_REPORT = Path("/content/stage6_colab_release_validation.json")
subprocess.run(
    [str(PYTHON), "scripts/release_check.py", "--report", str(RELEASE_REPORT)],
    check=True,
)
release = json.loads(RELEASE_REPORT.read_text(encoding="utf-8"))
assert release["status"] == "PASS", release["errors"]
assert len(release["deterministic_hashes"]) == 142
assert any(key.startswith("stage6_baseline/") for key in release["deterministic_hashes"])
print("PASS: canonical release gate and 142 deterministic outputs")


In [ ]:
# Generate a separate Stage 6 comparison and regenerate its candidate paper assets.
OUTPUTS = Path("/content/stage6_checkpoint_outputs")
ASSETS = Path("/content/stage6_checkpoint_paper_assets")
for path in (OUTPUTS, ASSETS):
    if path.exists():
        shutil.rmtree(path)
subprocess.run(
    [str(PYTHON), "scripts/run_baseline_comparison.py", "--output-root", str(OUTPUTS)],
    check=True,
)
subprocess.run(
    [
        str(PYTHON), "paper_assets/scripts/build_stage6_assets.py",
        "--output-root", str(OUTPUTS), "--destination", str(ASSETS),
    ],
    check=True,
)


In [ ]:
# Verify the frozen comparison design and the controlled pilot results.
REPORT = OUTPUTS / "comparison/stage6_baseline_report.json"
comparison = json.loads(REPORT.read_text(encoding="utf-8"))
metrics = {row["metric"]: row for row in comparison["primary_metrics"]}
assert comparison["scenario_count"] == 7
assert comparison["primary_scenario_count"] == 4
assert comparison["control_count"] == 3
assert comparison["evaluation_status"] == "PILOT_BASELINE_NOT_FROZEN"
assert comparison["manuscript_eligible"] is False
assert metrics["EC"]["orchestrated_primary_mean"] == 1.0
assert metrics["EC"]["baseline_primary_mean"] == 0.764706
assert metrics["TR"]["orchestrated_primary_mean"] == 1.0
assert metrics["TR"]["baseline_primary_mean"] == 0.0
assert metrics["CD"]["orchestrated_primary_mean"] == 1.0
assert metrics["CD"]["baseline_primary_mean"] == 1.0
assert metrics["CA"]["orchestrated_primary_mean"] == 1.0
assert metrics["CA"]["baseline_primary_mean"] == 0.0
assert metrics["AR"]["orchestrated_primary_mean"] == 1.0
assert metrics["AR"]["baseline_primary_mean"] == 1.0
assert metrics["SC"]["baseline_primary_mean"] == 0.708333
assert metrics["EPG"]["orchestrated_primary_mean"] == 1.0
assert metrics["EPG"]["baseline_primary_mean"] == 0.0
assert comparison["state_divergence_count"] == 5
assert comparison["conflict_precision"]["orchestrated"] == 1.0
assert comparison["conflict_precision"]["baseline"] == 0.5
assert comparison["source_review_operation_proxy"]["orchestrated_unique_source_ingestions"] == 86
assert comparison["source_review_operation_proxy"]["baseline_cumulative_source_reviews"] == 273
print("PASS: matched seven-replay comparison")
print("PASS: ties retained for CD and AR")
print("PASS: five mechanism-level state divergences")
print("PASS: baseline limitations and non-eligibility preserved")


In [ ]:
# Display the generated comparison figure.
from IPython.display import SVG, display
display(SVG(filename=str(ASSETS / "figures/stage6_metric_comparison.svg")))


In [ ]:
# Preserve exact code, authoritative EPSS evidence, comparison outputs, and assets.
import datetime as dt
import hashlib
import zipfile

environment = {
    "checked_at_utc": dt.datetime.now(dt.timezone.utc).isoformat(),
    "repository": REPO_URL,
    "reference": REF,
    "git_commit": COMMIT,
    "kernel_python": sys.version,
    "isolated_python": subprocess.check_output([str(PYTHON), "--version"], text=True).strip(),
    "platform": platform.platform(),
    "release_status": release["status"],
    "historical_epss_status": verification["status"],
    "baseline_status": comparison["evaluation_status"],
}
ENV = Path("/content/stage6_colab_environment.json")
ENV.write_text(json.dumps(environment, indent=2) + "\n", encoding="utf-8")
BUNDLE = Path("/content/stage6_colab_checkpoint_evidence.zip")
with zipfile.ZipFile(BUNDLE, "w", zipfile.ZIP_DEFLATED) as archive:
    archive.write(RELEASE_REPORT, RELEASE_REPORT.name)
    archive.write(ENV, ENV.name)
    for root in (EPSS_DIR, OUTPUTS, ASSETS):
        for path in sorted(root.rglob("*")):
            if path.is_file():
                archive.write(path, f"{root.name}/{path.relative_to(root)}")
digest = hashlib.sha256(BUNDLE.read_bytes()).hexdigest()
print("Created:", BUNDLE)
print("Tested Git commit:", COMMIT)
print("Colab evidence-bundle SHA-256:", digest)
print("Download the ZIP from Colab and preserve it unchanged.")
